# Datathon Passos Mágicos — Pergunta 9 (ML)
## Modelo preditivo de risco de defasagem

Este notebook demonstra as etapas de:
- Feature engineering
- Separação treino/teste
- Modelagem preditiva (pipeline)
- Avaliação de resultados
- Salvamento do modelo para uso na aplicação (Streamlit)

## Imports

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)

import joblib

## Path

REPO_ROOT = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "raw" / "BASE_DE_DADOS_PEDE_2024_DATATHON.xlsx"
MODEL_DIR = REPO_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

## carregar dados

df = pd.read_excel(DATA_PATH)
df.shape, df.columns.tolist()[:10]

## Selecao de colunas

required = ["Defas", "IDA", "IEG", "IAA", "IPS", "Fase", "Pedra 22", "Ano ingresso", "Idade 22", "Gênero", "Turma"]
missing = [c for c in required if c not in df.columns]
missing

## Conversao numerica

num_cols = ["IDA", "IEG", "IAA", "IPS", "Ano ingresso", "Idade 22"]
df["Defas"] = pd.to_numeric(df["Defas"], errors="coerce")
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[num_cols + ["Defas"]].describe().T

## definicao do alvo

## Definição do alvo (risco)

Como `Defas` é altamente concentrada em zero (evento raro), adotamos:
- **risco_defasagem = 1 se Defas > 0** (qualquer defasagem)

Isso torna o target robusto e útil para identificar precocemente o início de defasagem.

## Criar Target

df["risco_defasagem"] = (df["Defas"] > 0).astype(int)
df["risco_defasagem"].value_counts(normalize=True)

In [ ]:
## Feature Engineering

# Remover colunas que vazam o alvo
leak_cols = [c for c in ["Defas", "IAN"] if c in df.columns]

df_model = df.drop(columns=leak_cols).dropna(subset=["risco_defasagem"])

cat_cols = ["Fase", "Pedra 22", "Gênero", "Turma"]

X = df_model[num_cols + cat_cols]
y = df_model["risco_defasagem"]

X.head(), y.mean()

In [ ]:
## Predicao


p_test = pipe.predict_proba(X_test)[:, 1]
y_pred = (p_test >= 0.5).astype(int)

roc = roc_auc_score(y_test, p_test)
pr_auc = average_precision_score(y_test, p_test)

roc, pr_auc

In [ ]:
## Classification

print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
## roc

fpr, tpr, _ = roc_curve(y_test, p_test)
plt.figure()
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve — Risco de Defasagem")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.show()

In [ ]:
## PRecision

prec, rec, _ = precision_recall_curve(y_test, p_test)
plt.figure()
plt.plot(rec, prec)
plt.title("Precision-Recall Curve — Risco de Defasagem")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

In [ ]:
## Coeficientes

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = list(ohe.get_feature_names_out(cat_cols))
feature_names = num_cols + cat_feature_names

coefs = pipe.named_steps["clf"].coef_.ravel()

imp = pd.DataFrame({"feature": feature_names, "coef": coefs})
imp["abs_coef"] = imp["coef"].abs()
imp.sort_values("abs_coef", ascending=False).head(20)



In [ ]:
## DSalvar modelo

MODEL_PATH = MODEL_DIR / "q9_model.pkl"
META_PATH = MODEL_DIR / "q9_metadata.json"

joblib.dump(pipe, MODEL_PATH)

metadata = {
    "target": "risco_defasagem",
    "target_definition": "1 se Defas > 0 (qualquer defasagem)",
    "positive_rate_model": float(y.mean()),
    "features_numeric": num_cols,
    "features_categorical": cat_cols,
    "removed_leakage_cols": leak_cols,
    "roc_auc": float(roc),
    "pr_auc": float(pr_auc),
}
META_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

MODEL_PATH, META_PATH


## COnclusao

## Conclusão

- O evento “defasagem” é raro no dataset, portanto o problema é altamente desbalanceado.
- O pipeline com Logistic Regression e `class_weight="balanced"` permite identificar casos de risco com foco em recall.
- O modelo foi salvo em `models/q9_model.pkl` e pode ser utilizado na aplicação Streamlit.